# Player Rating Forecasting

Predict a football player's next-match rating from their recent form, rest, venue and position — with calibrated
90% intervals, SHAP explanations, "what-if" scenarios, similar historical matches, and a Streamlit dashboard.

**Pipeline:** load & clean → feature engineering → chronological split → baselines → two-model (starter / sub) system →
SHAP → conformal prediction intervals → match report & similarity search → export artifacts → Streamlit app.

Run the notebook top to bottom. Expects `ratings.csv` in the working directory.

## 1. Setup

In [ ]:
!pip install -q shap streamlit

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

DATA_PATH = "ratings.csv"
SEED = 42

# Chronological split: train <= 2020 | calibration (conformal) = 2021 | test >= 2022
TRAIN_END_YEAR = 2020
CALIB_YEAR = 2021
TEST_START_YEAR = 2022

ALPHA = 0.10  # 90% prediction intervals

FORM_FEATURES = [
    "rating_last_3",
    "rating_last_5",
    "rating_last_10",
    "days_since_last_match",
    "rating_std_5",
    "momentum",
]

## 2. Load & inspect data

In [ ]:
player_data = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")

print("Dataset shape (rows, columns):", player_data.shape)
print("Columns:", player_data.columns.tolist())
display(player_data.head(3))

print("\nRating statistics:")
print(player_data["rating"].describe())
print(f"\nMissing ratings (unused subs): {player_data['rating'].isnull().sum()}")
print(f"Exact 0.0 ratings:             {(player_data['rating'] == 0).sum()}")

## 3. Clean data

In [ ]:
# Drop rows with no rating, parse dates, and sort each player's matches chronologically
df_clean = (
    player_data.dropna(subset=["rating"])
    .assign(datetime=lambda d: pd.to_datetime(d["datetime"]))
    .sort_values(["player", "datetime"])
    .reset_index(drop=True)
)

match_counts = df_clean["player"].value_counts()
print("Cleaned dataset shape:", df_clean.shape)
print("Unique players:", len(match_counts))
print("Median matches per player:", match_counts.median())
print("\nPlayers with most data:")
print(match_counts.head())

## 4. Feature engineering

All rolling features use `shift(1)` so a match's features only ever see *earlier* matches (no target leakage).
Rows without enough history are dropped. `position` is one-hot encoded; `position_Sub` is converted into an
`is_starter` flag that is used to route rows to the starter / substitute models later.

In [ ]:
def past_rolling(df, window, stat, min_periods):
    # Rolling statistic over a player's *previous* matches only (shift(1) excludes the current one).
    return df.groupby("player")["rating"].transform(
        lambda s: getattr(s.shift(1).rolling(window, min_periods=min_periods), stat)()
    )


def build_features(df):
    df = df.sort_values(["player", "datetime"]).copy()

    # Form
    df["rating_last_3"] = past_rolling(df, 3, "mean", 1)
    df["rating_last_5"] = past_rolling(df, 5, "mean", 1)
    df["rating_last_10"] = past_rolling(df, 10, "mean", 5)
    df["rating_std_5"] = past_rolling(df, 5, "std", 3)                      # volatility
    df["momentum"] = df["rating_last_3"] - df["rating_last_10"]             # short- vs long-term form

    # Workload proxy
    df["days_since_last_match"] = df.groupby("player")["datetime"].diff().dt.days

    # Context
    df["is_home"] = (df["home_or_away"].astype(str).str.lower() == "home").astype(int)
    df["opponent"] = np.where(df["is_home"] == 1, df["away_team"], df["home_team"])
    df["year"] = df["datetime"].dt.year

    # Drop each player's first matches (not enough history)
    df = df.dropna(subset=["rating_last_10", "days_since_last_match", "rating_std_5"])

    # Position one-hot; position_Sub -> is_starter
    df = pd.get_dummies(df, columns=["position"], drop_first=True, dtype=int)
    df["is_starter"] = (df["position_Sub"] == 0).astype(int)
    return df.drop(columns="position_Sub").reset_index(drop=True)


df_master = build_features(df_clean)

POSITION_COLUMNS = [c for c in df_master.columns if c.startswith("position_")]
ALL_FEATURES = FORM_FEATURES + ["is_home"] + POSITION_COLUMNS

print(df_master["year"].value_counts().sort_index())

# Leakage sanity check: rolling features should lag the ratings column
display(
    df_master.loc[
        df_master["player"] == "Luis Suárez",
        ["datetime", "rating", "rating_last_3", "rating_last_5", "rating_last_10"],
    ].head(7)
)

## 5. Chronological split

In [ ]:
train = df_master[df_master["year"] <= TRAIN_END_YEAR]
calib = df_master[df_master["year"] == CALIB_YEAR]
test = df_master[df_master["year"] >= TEST_START_YEAR]

print(f"Train: {len(train):,} | Calibration: {len(calib):,} | Test: {len(test):,}")

## 6. Baselines & contextual model

In [ ]:
def make_xgb(max_depth=5):
    return xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=max_depth, random_state=SEED)


def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


results = {}

# Naive: predict the last-5-match average
results["Naive (last-5 mean)"] = mae(test["rating"], test["rating_last_5"])

# Linear regression on form features
lr_model = LinearRegression().fit(train[FORM_FEATURES], train["rating"])
results["Linear regression"] = mae(test["rating"], lr_model.predict(test[FORM_FEATURES]))

# XGBoost on form features only
xgb_form = make_xgb().fit(train[FORM_FEATURES], train["rating"])
results["XGBoost (form features)"] = mae(test["rating"], xgb_form.predict(test[FORM_FEATURES]))

# XGBoost with context (venue + position + starter flag)
context_features = ALL_FEATURES + ["is_starter"]
xgb_context = make_xgb().fit(train[context_features], train["rating"])
results["XGBoost (+ context)"] = mae(test["rating"], xgb_context.predict(test[context_features]))

print("Linear regression weights:")
for name, coef in zip(FORM_FEATURES, lr_model.coef_):
    print(f"  {name:24}: {coef:+.4f}")

print("\nXGBoost (+ context) top features:")
display(
    pd.DataFrame({"Feature": context_features, "Importance": xgb_context.feature_importances_})
    .sort_values("Importance", ascending=False)
    .head(8)
)

## 7. Two-model system (starters vs substitutes)

Whether a player starts is by far the strongest signal, so starters and substitutes get separate models.
Substitutes use a shallower tree because their ratings cluster near the default (~6.0).

In [ ]:
def split_by_role(df):
    return df[df["is_starter"] == 1], df[df["is_starter"] == 0]


def top_features(model, features, n=3):
    return (
        pd.DataFrame({"Feature": features, "Importance": model.feature_importances_})
        .sort_values("Importance", ascending=False)
        .head(n)
    )


train_starters, train_subs = split_by_role(train)
test_starters, test_subs = split_by_role(test)

model_starters = make_xgb(max_depth=5).fit(train_starters[ALL_FEATURES], train_starters["rating"])
model_subs = make_xgb(max_depth=3).fit(train_subs[ALL_FEATURES], train_subs["rating"])

mae_starters = mae(test_starters["rating"], model_starters.predict(test_starters[ALL_FEATURES]))
mae_subs = mae(test_subs["rating"], model_subs.predict(test_subs[ALL_FEATURES]))
combined_mae = (mae_starters * len(test_starters) + mae_subs * len(test_subs)) / len(test)
results["Two models (starters + subs)"] = combined_mae

print(f"Starters MAE: {mae_starters:.4f}")
print(f"Subs MAE:     {mae_subs:.4f}")
print(f"Combined MAE: {combined_mae:.4f}\n")

print("Top features - starters:")
display(top_features(model_starters, ALL_FEATURES))
print("Top features - substitutes:")
display(top_features(model_subs, ALL_FEATURES))

print("Test MAE by approach:")
display(pd.Series(results, name="Test MAE").round(4).to_frame())

## 8. Explainability (SHAP)

In [ ]:
explainer_starters = shap.TreeExplainer(model_starters)
explainer_subs = shap.TreeExplainer(model_subs)

# Sample for speed
X_shap = test_starters[ALL_FEATURES].sample(1000, random_state=SEED)
shap_values = explainer_starters(X_shap)

# Global: how feature values push predictions up or down
shap.summary_plot(shap_values, X_shap, show=False)
plt.title("SHAP Summary: Starters")
plt.show()

# Local: one match, from base value to final prediction
base_value = float(shap_values[0].base_values)
prediction = float(model_starters.predict(X_shap.iloc[[0]])[0])
print(f"Base expected rating for an average starter: {base_value:.2f}")
print(f"Model prediction for this match:             {prediction:.2f}")
shap.plots.waterfall(shap_values[0], max_display=10)

## 9. Conformal prediction intervals

Split-conformal calibration on 2021 starters gives a margin `q_hat`; intervals are `prediction ± q_hat`,
clipped to [0, 10]. Coverage is verified on the held-out test set.

In [ ]:
def conformal_margin(residuals, alpha):
    # Finite-sample-corrected quantile of absolute calibration errors
    n = len(residuals)
    level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return np.quantile(residuals, level, method="higher")


calib_starters, _ = split_by_role(calib)
calib_residuals = np.abs(
    calib_starters["rating"].values - model_starters.predict(calib_starters[ALL_FEATURES])
)
q_hat = conformal_margin(calib_residuals, ALPHA)

print(f"Calibration size ({CALIB_YEAR} starters): {len(calib_residuals):,}")
print(f"Conformal margin (q_hat): +/-{q_hat:.3f}")

# Empirical coverage on the test set
test_preds = model_starters.predict(test_starters[ALL_FEATURES])
lower = np.clip(test_preds - q_hat, 0, 10)
upper = np.clip(test_preds + q_hat, 0, 10)
actual = test_starters["rating"].values
coverage = np.mean((actual >= lower) & (actual <= upper)) * 100

print(f"Nominal coverage:   {100 * (1 - ALPHA):.1f}%")
print(f"Empirical coverage: {coverage:.2f}%")

## 10. Match report & similar historical matches

Console prototypes of the dashboard features (output text is in French, matching the app).

In [ ]:
def generate_match_report(model, explainer, data_row, features, q_hat, hist_mae):
    row_df = pd.DataFrame([data_row[features]]).astype(float)

    # Prediction + interval
    pred = model.predict(row_df)[0]
    low, high = np.clip(pred - q_hat, 0, 10), np.clip(pred + q_hat, 0, 10)

    # SHAP: top drivers by absolute impact
    shap_df = pd.DataFrame(
        {"Feature": row_df.columns, "Impact": explainer(row_df).values[0]}
    ).sort_values("Impact", key=abs, ascending=False)
    top_pos = shap_df[shap_df["Impact"] > 0].head(2)["Feature"].tolist()
    top_neg = shap_df[shap_df["Impact"] < 0].head(2)["Feature"].tolist()

    # What-if scenarios
    row_rest = row_df.copy()
    row_rest["days_since_last_match"] = 7.0
    pred_rest = model.predict(row_rest)[0]

    row_venue = row_df.copy()
    row_venue["is_home"] = 1.0 - row_df["is_home"].iloc[0]
    pred_venue = model.predict(row_venue)[0]
    venue_str = "l'Extérieur" if row_df["is_home"].iloc[0] == 1 else "Domicile"

    line = "=" * 50
    thin = "-" * 50
    print(line)
    print(f"JOUEUR :           {data_row['player']}")
    print(f"DATE DU MATCH :    {data_row['datetime'].strftime('%Y-%m-%d')}")
    print(line)
    print(f"NOTE PRÉDITE :     {pred:.2f} / 10.0")
    print(f"PLAGE ATTENDUE (90 %) : [{low:.2f} — {high:.2f}]")
    print(f"FIABILITÉ :        Calibrée (MAE Historique : {hist_mae:.2f})")
    print(thin)
    print(f"FORME RÉCENTE (L5) : {data_row['rating_last_5']:>4.2f}")
    print(f"LONG TERME (L10) :   {data_row['rating_last_10']:>4.2f}")
    print(f"JOURS DE REPOS :     {data_row['days_since_last_match']:>4.0f}")
    print(thin)
    print("FACTEURS POSITIFS CLÉS :")
    for f in top_pos or ["Aucun facteur significatif"]:
        print(f" [+] {f}")
    print("\nFACTEURS NÉGATIFS CLÉS :")
    for f in top_neg or ["Aucun facteur significatif"]:
        print(f" [-] {f}")
    print(thin)
    print("SCÉNARIOS 'ET SI' (Selon les relations apprises par le modèle) :")
    print(f" * Avec 7 jours complets de repos :         {pred_rest:.2f} ({pred_rest - pred:+.2f})")
    print(f" * Si le match était joué à {venue_str:<11} : {pred_venue:.2f} ({pred_venue - pred:+.2f})")
    print(line)


# Similarity engine: standardise features (fit on train only), then nearest neighbours
scaler = StandardScaler().fit(train_starters[ALL_FEATURES])
similarity_engine = NearestNeighbors(n_neighbors=3, metric="euclidean").fit(
    scaler.transform(train_starters[ALL_FEATURES])
)


def find_similar_matches(data_row, features, engine, scaler, historical_df):
    current = scaler.transform(data_row[features].astype(float).to_frame().T)
    _, indices = engine.kneighbors(current)

    line = "=" * 50
    print(line)
    print(f"RECHERCHE DE MATCHS HISTORIQUES SIMILAIRES POUR : {data_row['player']}")
    print(line)

    ratings = []
    for i, idx in enumerate(indices[0], start=1):
        hist = historical_df.iloc[idx]
        ratings.append(hist["rating"])
        print(f"MATCH SIMILAIRE #{i} : {hist['player']} (Date: {hist['datetime'].strftime('%Y-%m-%d')})")
        print(f"  * Forme (L5) :      {hist['rating_last_5']:.2f}")
        print(f"  * Jours de repos :  {hist['days_since_last_match']:.0f}")
        print(f"  * Domicile/Ext. :   {'Domicile' if hist['is_home'] == 1 else 'Extérieur'}")
        print(f"  -> NOTE OBTENUE :   {hist['rating']:.2f}")
        print("-" * 50)
    print(f"MOYENNE DES MATCHS SIMILAIRES : {np.mean(ratings):.2f}")
    print(line)


sample_row = test_starters.iloc[0]
generate_match_report(model_starters, explainer_starters, sample_row, ALL_FEATURES, q_hat, mae_starters)
find_similar_matches(sample_row, ALL_FEATURES, similarity_engine, scaler, train_starters)

## 11. Export artifacts for the dashboard

The app's fixture list is every starter row from the test period (2022+), with the opponent attached.

In [ ]:
joblib.dump(model_starters, "model_starters.pkl")
joblib.dump(model_subs, "model_subs.pkl")
joblib.dump(similarity_engine, "similarity_engine.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump({"q_hat": q_hat, "features": ALL_FEATURES}, "config.pkl")

# Fixtures the dashboard can forecast
(
    test_starters.sort_values(["player", "datetime"])[["player", "datetime", "opponent"] + ALL_FEATURES]
    .to_csv("upcoming_matches.csv", index=False)
)

# History searched by the similarity engine (same row order as the engine was fit on)
train_starters[
    ["player", "datetime", "rating_last_5", "days_since_last_match", "is_home", "rating"]
].to_csv("historical_matches.csv", index=False)

print("All artifacts exported.")

## 12. Streamlit dashboard

In [ ]:
%%writefile app.py
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import streamlit as st

VENUE_LABEL = {1: "Domicile", 0: "Extérieur"}

# --- 1. CONFIGURATION & LOADING ---
st.set_page_config(page_title="Prévision de Performance", layout="wide")


@st.cache_resource
def load_artifacts():
    model_starters = joblib.load("model_starters.pkl")
    model_subs = joblib.load("model_subs.pkl")
    knn = joblib.load("similarity_engine.pkl")
    scaler = joblib.load("scaler.pkl")
    config = joblib.load("config.pkl")
    explainers = {
        "starter": shap.TreeExplainer(model_starters),
        "sub": shap.TreeExplainer(model_subs),
    }
    models = {"starter": model_starters, "sub": model_subs}
    return models, explainers, knn, scaler, config


models, explainers, knn, scaler, config = load_artifacts()
q_hat = config["q_hat"]
features = config["features"]

df_upcoming = pd.read_csv("upcoming_matches.csv")
df_history = pd.read_csv("historical_matches.csv")

# --- 2. SIDEBAR SELECTION ---
st.sidebar.header("Paramètres de Prévision")

selected_player = st.sidebar.selectbox("1. Choisir un joueur :", sorted(df_upcoming["player"].unique()))
role = st.sidebar.radio("2. Statut pour ce match :", ["Titulaire (Starter)", "Remplaçant (Substitute)"])

player_fixtures = (
    df_upcoming[df_upcoming["player"] == selected_player]
    .sort_values("datetime", ascending=False)
    .reset_index(drop=True)
)


def fixture_label(i):
    row = player_fixtures.iloc[i]
    opponent = row["opponent"] if pd.notna(row["opponent"]) else "Adversaire"
    return f"{row['datetime']} - vs {opponent} ({VENUE_LABEL[row['is_home']]})"


selected_idx = st.sidebar.selectbox(
    "3. Choisir le match à prévoir :", range(len(player_fixtures)), format_func=fixture_label
)
player_row = player_fixtures.iloc[selected_idx]
X_input = player_fixtures.iloc[[selected_idx]][features].astype(float)

# --- 3. PREDICTIONS ---
if role == "Titulaire (Starter)":
    key, role_color = "starter", "blue"
else:
    key, role_color = "sub", "orange"
active_model, active_explainer = models[key], explainers[key]

base_pred = float(active_model.predict(X_input)[0])
interval_low = np.clip(base_pred - q_hat, 0, 10)
interval_high = np.clip(base_pred + q_hat, 0, 10)

# Dynamic reliability label (based on interval width)
interval_width = interval_high - interval_low
if interval_width < 1.5:
    reliability = "🟢 Haute"
elif interval_width < 2.5:
    reliability = "🟡 Moyenne"
else:
    reliability = "🔴 Faible"

# --- 4. MAIN DASHBOARD UI ---
st.title("⚽ Tableau de Bord : Prévision de Performance")
st.markdown(
    f"**Joueur :** `{selected_player}` | "
    f"**Rôle :** :{role_color}[{role}] | "
    f"**Date :** `{player_row['datetime']}` | "
    f"**Adversaire :** `{player_row.get('opponent', 'N/A')}`"
)
st.divider()

col1, col2, col3 = st.columns(3)
col1.metric("Note Prédite", f"{base_pred:.2f} / 10")
col2.metric("Intervalle de Confiance (90%)", f"[{interval_low:.2f} — {interval_high:.2f}]")
col3.metric("Indice de Fiabilité", reliability, f"Marge : ±{q_hat:.2f} pts", delta_color="off")
st.divider()

col_left, col_right = st.columns(2)

# --- 5. EXPLICABILITÉ (TEXT & SHAP) ---
with col_left:
    st.subheader("📊 Facteurs d'Impact (Analyse)")

    shap_vals = active_explainer(X_input)
    impacts = sorted(zip(features, shap_vals[0].values), key=lambda x: x[1], reverse=True)
    pos_factors = [f for f in impacts if f[1] > 0][:3]
    neg_factors = [f for f in reversed(impacts) if f[1] < 0][:3]

    st.markdown("**Points Positifs Clés :**")
    if pos_factors:
        for feat, val in pos_factors:
            st.markdown(f"- 🟢 **{feat}** (+{val:.2f} pts)")
    else:
        st.markdown("- Aucun facteur positif majeur.")

    st.markdown("**Points Négatifs Clés :**")
    if neg_factors:
        for feat, val in neg_factors:
            st.markdown(f"- 🔴 **{feat}** ({val:.2f} pts)")
    else:
        st.markdown("- Aucun facteur négatif majeur.")

    st.markdown("<br>", unsafe_allow_html=True)
    fig, ax = plt.subplots(figsize=(5, 3))
    shap.plots.waterfall(shap_vals[0], max_display=5, show=False)
    st.pyplot(fig)
    plt.close(fig)

# --- 6. WHAT-IF COUNTERFACTUALS ---
with col_right:
    st.subheader("🔄 Analyse Contrefactuelle (Et si ?)")

    X_rest = X_input.copy()
    X_rest["days_since_last_match"] += 3
    pred_rest = float(active_model.predict(X_rest)[0])
    st.write(
        f"**Si le joueur disposait de 3 jours de repos en plus :** "
        f"{pred_rest:.2f} ({pred_rest - base_pred:+.2f})"
    )

    X_venue = X_input.copy()
    X_venue["is_home"] = 1.0 - X_venue["is_home"].iloc[0]
    other_venue = VENUE_LABEL[1 - int(player_row["is_home"])]
    pred_venue = float(active_model.predict(X_venue)[0])
    st.write(f"**Si le match se jouait à {other_venue} :** {pred_venue:.2f} ({pred_venue - base_pred:+.2f})")

st.divider()

# --- 7. SIMILAR HISTORICAL MATCHES ---
st.subheader("🔍 Matchs Historiques Similaires (Contexte Comparé)")
_, indices = knn.kneighbors(scaler.transform(X_input))

st.dataframe(
    df_history.iloc[indices[0]][["player", "datetime", "rating_last_5", "days_since_last_match", "rating"]]
    .rename(columns={
        "player": "Joueur",
        "datetime": "Date",
        "rating_last_5": "Forme Récente (L5)",
        "days_since_last_match": "Jours Repos",
        "rating": "Note Obtenue",
    })
    .reset_index(drop=True),
    width="stretch",
)

## 13. Launch the dashboard (Colab)

In [ ]:
import time
from google.colab.output import eval_js

!pkill -f streamlit
!nohup python -m streamlit run app.py --server.address 0.0.0.0 --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1 &
time.sleep(5)

print("Dashboard:", eval_js("google.colab.kernel.proxyPort(8501)"))
print("If it doesn't load, check the server log:  !cat streamlit.log")